# 05 — Chronological split and leakage audit

This notebook reads the frozen Mission 02 split manifest generated from the verified ULB CSV. It does not train a model. Complete equal-`Time` groups are kept together; the final 15% is reserved for the final evaluation. The manifest contains aggregate counts and can be regenerated from the locally verified source.

`Class` appears only in partition summaries and the duplicate audit. It is not in the model feature list.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
manifest = json.loads((ROOT / "reports/ml/split_manifest.json").read_text(encoding="utf-8"))
partitions = pd.DataFrame.from_dict(manifest["partitions"], orient="index")
partitions["fraud_count"] = partitions["class_counts"].map(lambda value: value["1"])
partitions["legitimate_count"] = partitions["class_counts"].map(lambda value: value["0"])
display(
    partitions[
        ["rows", "legitimate_count", "fraud_count", "fraud_percentage", "time_min", "time_max"]
    ]
)
print("Manifest ID:", manifest["manifest_id"])
print("Dataset SHA-256:", manifest["dataset_sha256"])
print("Duplicate audit:", manifest["duplicate_audit"])
print("Features include Class:", "Class" in manifest["feature_columns"])

fig, ax = plt.subplots(figsize=(7, 4))
partitions["rows"].plot.bar(ax=ax, color="#3768a0", title="Rows in frozen chronological partitions")
ax.set_ylabel("Rows")
ax.set_xlabel("Partition")
plt.tight_layout()
plt.show()